# Education institutions, individuals and employers

A separate, reproducible analysis of the supplied education and career files, with birth-year/gender attributes from main. Run all cells from this folder after installing requirements.txt. The original analysis is preserved separately.

**Interpretation:** the files do not contain appointment dates. A qualification may precede or follow a job. Graphs and flows show recorded associations, not causal or confirmed chronological education-to-employment transitions. “Current” refers to the historical source. Unknown categories are retained.

[Methods and outputs](methods.html) · [Individual/institution network](individual_institutions.html) · [University–employer projection](university_employer.html) · [Multistage Sankey](education_employment_sankey.html)

## 1. Read and validate sources

The main and career files use semicolons; education uses commas. Validate unique individuals and appointment IDs before linking. Original inputs and hashes are preserved.

In [1]:
from pathlib import Path
import json, hashlib, itertools
import pandas as pd
import numpy as np
import networkx as nx

ROOT=Path.cwd()
if not (ROOT/'data/main.csv').exists(): ROOT=Path(__file__).resolve().parent
TABLES=ROOT/'tables'; TABLES.mkdir(exist_ok=True)
UNKNOWN='Unknown'
def read(name,sep=','):
    d=pd.read_csv(ROOT/'data'/name,sep=sep)
    d=d.loc[:,~d.columns.str.startswith('Unnamed:')].copy()
    for c in d.select_dtypes('object'): d[c]=d[c].str.strip().replace('',np.nan)
    return d
main=read('main.csv',';');education=read('educ.csv');career=read('career.csv',';')
assert main.NameID.is_unique and main.NameID.notna().all()
assert career.record_id.is_unique and career.record_id.notna().all()
assert set(education.NameID)<=set(main.NameID)
assert set(career.NameID)<=set(main.NameID)
pd.DataFrame([{'file':p.name,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted((ROOT/'data').glob('*.csv'))]).to_csv(TABLES/'source_manifest.csv',index=False)
print(f'{len(main):,} people; {len(education):,} education rows; {len(career):,} appointment rows.')

1,613 people; 3,151 education rows; 1,689 appointment rows.


## 2. Prepare person, education and employment records

Education and appointments remain separate fact tables. Education filters use the source `Level` column as `education_level`; raw Degree is retained only in exports. University, country, discipline and education level must match the same education row. Study bounds use the full education history. Education coordinates prefer revised values, then original coordinates; workplace coordinates come from career. Invalid or missing coordinates remain unavailable rather than being inferred.

In [2]:
education['Year']=pd.to_numeric(education.Year,errors='coerce')
span=education.groupby('NameID').Year.agg(study_start='min',study_end='max')
people=main[['NameID','Gender','birth_year','Birth_province_py_st']].merge(span,on='NameID',how='left',validate='one_to_one').rename(columns={'Gender':'gender'})
names=career.groupby('NameID').NameFull.first().combine_first(education.groupby('NameID').FullName_PY.first())
people['label']=people.NameID.map(lambda p:p+' · '+str(names.get(p,p)))
people.gender=people.gender.fillna(UNKNOWN)
people['birth_decade']=people.birth_year.map(lambda y:f'{int(y)//10*10}s' if pd.notna(y) else UNKNOWN)
people['study_start_decade']=people.study_start.map(lambda y:f'{int(y)//10*10}s' if pd.notna(y) else UNKNOWN)
edu=education[['NameID','University_clean','University','Country','discipline','Degree','Level','Year','City','Province_State']].copy()
edu['lat']=education.latitude_revised.combine_first(education.Latitude)
edu['lon']=education.longitude_revised.combine_first(education.Longitude)
edu['education_id']=[f'ED{i+1:05d}' for i in range(len(edu))]
edu['university_label_source']=np.where(edu.University_clean.notna(),'University_clean',np.where(edu.University.notna(),'University','missing'))
edu['University_clean']=edu.University_clean.combine_first(edu.University)
edu=edu.drop(columns='University').rename(columns={'University_clean':'university','Country':'university_country','Degree':'degree','Level':'education_level','Year':'year','City':'city','Province_State':'province'})
jobs=career[['record_id','NameID','Institution_normalized','Country_completed','institution_sector','advancement_level_valid','appointment_status_normalized','Position_normalized']].rename(columns={'Institution_normalized':'employer','Country_completed':'employer_country','institution_sector':'sector','advancement_level_valid':'level','appointment_status_normalized':'status','Position_normalized':'position'})
jobs['city']=career.City_completed;jobs['province']=career.State_province_completed
jobs['lat']=career.latitude;jobs['lon']=career.longitude
for d,cols in [(edu,['university','university_country','discipline','degree','education_level','city','province']),(jobs,['employer','employer_country','sector','level','status','position','city','province'])]:
    for c in cols:d[c]=d[c].fillna(UNKNOWN)
for d in [edu,jobs]:d['valid_coordinates']=d.lat.between(-90,90)&d.lon.between(-180,180)
edu.university_country=edu.university_country.replace({'USA':'United States'})
jobs.employer_country=jobs.employer_country.replace({'USA':'United States'})
jobs.status=jobs.status.replace({'Former':'Past'})
def export(d,name):d.to_csv(TABLES/(name+'.csv'),index=False,encoding='utf-8-sig')
export(people,'students_synthetic');export(edu,'education_normalized');export(jobs,'employment_normalized')
export(edu.merge(people,on='NameID',validate='many_to_one'),'education_with_person_attributes')
export(jobs.merge(people,on='NameID',validate='many_to_one'),'employment_with_person_attributes')
export(edu.loc[edu.university_label_source.ne('University_clean')],'university_label_fallbacks')
print('University labels prefer University_clean, then University. Unknown institution labels cannot identify a network node.')

University labels prefer University_clean, then University. Unknown institution labels cannot identify a network node.


In [3]:
people[["NameID","gender","birth_year","study_start","study_end"]].head(10)

,NameID,gender,birth_year,study_start,study_end
0,N00001,F,1918.0,1943.0,1943.0
1,N00002,F,NaN,1943.0,1943.0
2,N00003,M,1919.0,1939.0,1943.0
3,N00004,F,NaN,1943.0,1943.0
4,N00005,M,NaN,1940.0,1942.0
5,N00006,M,NaN,1943.0,1943.0
6,N00007,F,NaN,1943.0,1943.0
7,N00008,M,NaN,1940.0,1940.0
8,N00009,M,NaN,1941.0,1943.0
9,N00010,M,1918.0,1943.0,1943.0


## 3. Individuals and institutions: three and two modes

Require at least one known matching university and employer per person. This matched population is retained across the three-mode and the two person–institution views. One distinct affiliation creates one edge; raw record repetition cannot inflate it. Degree centrality is normalized by the permitted opposite modes, then range thresholds select the displayed graph.

Both network exports now use text-element popups, with adjustable physics pace (0.25×–2×) and Pause/Resume. Controls preserve their state after filtering; retained nodes reuse their positions. Physics changes layout only, not graph counts or measures.

In [4]:
known_edu=edu.loc[edu.university.ne(UNKNOWN)]
known_jobs=jobs.loc[jobs.employer.ne(UNKNOWN)]
eligible=sorted(set(known_edu.NameID)&set(known_jobs.NameID))
em=known_edu.loc[known_edu.NameID.isin(eligible)].copy()
jm=known_jobs.loc[known_jobs.NameID.isin(eligible)].copy()
university_aff=em[['NameID','university']].drop_duplicates()
employer_aff=jm[['NameID','employer']].drop_duplicates()
export(university_aff,'individual_university_affiliations');export(employer_aff,'individual_employer_affiliations')
coverage=people[['NameID','gender']].copy()
coverage['has_known_university']=coverage.NameID.isin(known_edu.NameID)
coverage['has_known_employer']=coverage.NameID.isin(known_jobs.NameID)
coverage['eligible_for_combined_views']=coverage.NameID.isin(eligible)
export(coverage,'person_coverage')
universities=sorted(university_aff.university.unique());employers=sorted(employer_aff.employer.unique())
G=nx.Graph()
G.add_nodes_from(('p:'+p,{'kind':'person'}) for p in eligible)
G.add_nodes_from(('u:'+u,{'kind':'university'}) for u in universities)
G.add_nodes_from(('e:'+e,{'kind':'employer'}) for e in employers)
G.add_edges_from(('p:'+r.NameID,'u:'+r.university) for r in university_aff.itertuples())
G.add_edges_from(('p:'+r.NameID,'e:'+r.employer) for r in employer_aff.itertuples())
def graph_tables(graph,counts):
    nodes=[]
    for n,d in sorted(graph.nodes(data=True)):
        denominator=counts['university']+counts['employer'] if d['kind']=='person' else counts['person']
        nodes.append({'id':n,'kind':d['kind'],'degree':graph.degree(n),'centrality':graph.degree(n)/denominator if denominator else 0})
    return pd.DataFrame(nodes),pd.DataFrame([{'source':a,'target':b,'weight':1} for a,b in graph.edges()])
sizes={'person':len(eligible),'university':len(universities),'employer':len(employers)}
nodes,edges=graph_tables(G,sizes);export(nodes,'three_mode_nodes');export(edges,'three_mode_edges')
for kind in ['university','employer']:
    graph=G.subgraph([n for n,d in G.nodes(data=True) if d['kind'] in ['person',kind]]).copy()
    n,e=graph_tables(graph,{'person':len(eligible),'university':len(universities) if kind=='university' else 0,'employer':len(employers) if kind=='employer' else 0})
    export(n,'person_'+kind+'_nodes');export(e,'person_'+kind+'_edges')
summary={'source_people':len(people),'eligible_people':len(eligible),'universities':len(universities),'employers':len(employers),'education_records_retained':len(em),'appointments_retained':len(jm),'three_mode_nodes':G.number_of_nodes(),'three_mode_edges':G.number_of_edges(),'unknown_university_records':int(edu.university.eq(UNKNOWN).sum()),'unknown_employer_records':int(jobs.employer.eq(UNKNOWN).sum())}
print(json.dumps(summary,indent=2))

{
  "source_people": 1613,
  "eligible_people": 912,
  "universities": 301,
  "employers": 757,
  "education_records_retained": 1852,
  "appointments_retained": 1661,
  "three_mode_nodes": 1970,
  "three_mode_edges": 3358,
  "unknown_university_records": 2,
  "unknown_employer_records": 9
}


In [5]:
nodes.sort_values("centrality",ascending=False).head(15)

,id,kind,degree,centrality
1920,u:University of Michigan,university,94,0.103070
1961,u:Yenching University,university,89,0.097588
1776,u:Jiaotong University,university,82,0.089912
1895,u:Tsinghua University,university,74,0.081140
1796,u:Massachusetts Institute of Technology,university,74,0.081140
1721,u:Columbia University,university,69,0.075658
1757,u:Harvard University,university,56,0.061404
1819,u:New York University,university,50,0.054825
1902,u:University of California,university,49,0.053728
456,e:National Resources Commission,employer,47,0.051535


## 4. University–employer projection

Each person links their distinct universities to their distinct employers. Edge weight counts distinct shared NameIDs. This is a person-level affiliation cross-product, not a raw education-row × appointment-row job count, and not an observed chronological transition. University and employer roles remain distinct even for identical institution names.

In [6]:
# The affiliation cross-product is explicit and labelled. It is not a raw-record join.
pair_people={}
u_by=university_aff.groupby('NameID').university.agg(list)
e_by=employer_aff.groupby('NameID').employer.agg(list)
links_by_person=[]
for person in eligible:
    for u,e in itertools.product(u_by[person],e_by[person]):
        pair_people.setdefault((u,e),set()).add(person)
        links_by_person.append({'NameID':person,'university':u,'employer':e,'allocated_student_weight':1/(len(u_by[person])*len(e_by[person]))})
pair_rows=[{'university':u,'employer':e,'shared_students':len(ps),'NameIDs':'|'.join(sorted(ps))} for (u,e),ps in sorted(pair_people.items())]
projection_edges=pd.DataFrame(pair_rows)
P=nx.Graph()
P.add_nodes_from(('u:'+u,{'kind':'university'}) for u in universities)
P.add_nodes_from(('e:'+e,{'kind':'employer'}) for e in employers)
P.add_weighted_edges_from(('u:'+u,'e:'+e,len(ps)) for (u,e),ps in pair_people.items())
projection_nodes=pd.DataFrame([{'id':n,'label':n[2:],'kind':d['kind'],'degree':P.degree(n),'centrality':P.degree(n)/(len(employers) if d['kind']=='university' else len(universities)),'strength':P.degree(n,weight='weight')} for n,d in sorted(P.nodes(data=True))])
export(projection_nodes,'university_employer_nodes');export(projection_edges,'university_employer_edges')
export(pd.DataFrame(links_by_person),'university_employer_links_by_person')
summary['projection_edges']=P.number_of_edges()
print(f'{P.number_of_nodes():,} institution nodes and {P.number_of_edges():,} distinct university–employer pairs.')

1,058 institution nodes and 2,520 distinct university–employer pairs.


In [7]:
projection_edges.sort_values("shared_students",ascending=False).head(20)

,university,employer,shared_students,NameIDs
1688,Tsinghua University,Tsinghua University,26,N00027|N00044|N00130|N00135|N00156|N00331|N003...
2040,University of Michigan,University of Michigan,17,N00001|N00010|N00110|N00192|N00452|N00454|N004...
752,Jiaotong University,National Resources Commission,16,N00019|N00028|N00029|N00098|N00496|N00564|N006...
2443,Yenching University,Yenching University,14,N00110|N00123|N00471|N00490|N00624|N00684|N007...
1070,New York University,Bank of China,14,N00107|N00117|N00226|N00313|N00424|N00516|N005...
159,Central Army Military Academy,Department of Military Training,12,N00014|N00017|N00062|N00097|N00165|N00224|N004...
772,Jiaotong University,Universal Trading Corporation,11,N00034|N00086|N00328|N00333|N00521|N00649|N010...
1981,University of Michigan,Giffels & Vallet Incorporated,11,N00116|N00147|N00286|N00363|N00442|N00546|N007...
982,Massachusetts Institute of Technology,Universal Trading Corporation,10,N00287|N00337|N00339|N00522|N00567|N00649|N006...
977,Massachusetts Institute of Technology,Tsinghua University,10,N00016|N00130|N00331|N00375|N00396|N00716|N007...


## 5. Direct education institution → employer Sankey

Only two stages remain. Each institution pair is weighted by distinct bridging individuals after filters, with IDs and names available on hover and in exports. Repeated source rows cannot inflate a pair count. A person can bridge multiple pairs, so counts across links are not additive unique-person totals. Optional Other groups deduplicate people within each displayed grouped pair. Discipline and education level remain filters, with no discipline, degree or individual stages. These are recorded associations; appointment dates are unavailable.

In [8]:
labels=people.set_index('NameID').label.to_dict()
direct_links=pd.DataFrame([{'source':'University: '+u,'target':'Employer: '+e,'weight':len(ps),'individuals':len(ps),'NameIDs':json.dumps(sorted(ps),ensure_ascii=False),'individual_names':json.dumps([labels[p] for p in sorted(ps)],ensure_ascii=False)} for (u,e),ps in sorted(pair_people.items())])
export(direct_links,'sankey_direct_links')
for stale in ['sankey_individual_links.csv','sankey_education_detail_links.csv']:
    if (TABLES/stale).exists():(TABLES/stale).unlink()
print(f'Direct Sankey: {len(direct_links):,} university–employer pairs. Link weight is distinct bridging people; counts across pairs are not additive unique-person totals.')

Direct Sankey: 2,520 university–employer pairs. Link weight is distinct bridging people; counts across pairs are not additive unique-person totals.


## 6. Education-place → workplace flow map

Use matching records with valid coordinates on both sides. Group locations by country, province and city, with an unweighted mean of distinct venue coordinates as the reference point. Count each person once per city pair, even with repeated schools or jobs in the same city. Unknown cities with coordinates remain separate by institution/coordinate. Export mapped flows and missing-coordinate audits. The browser reproduces these calculations after filters, splits lines at the antimeridian and displays direction arrows. These are association lines rather than confirmed travel paths.

In [9]:
from flow_reference import build_flow_reference
flow_rows,flow_coverage=build_flow_reference(people.loc[people.NameID.isin(eligible)],em,jm)
export(flow_rows,'education_work_city_flows')
export(em.loc[~em.valid_coordinates],'unmapped_education_records')
export(jm.loc[~jm.valid_coordinates],'unmapped_employment_records')
export(pd.DataFrame(flow_coverage.items(),columns=['metric','value']),'map_coverage')
summary.update(flow_coverage)
(ROOT/'assets/world_data.js').write_text('window.WORLD='+ (ROOT/'assets/world.geojson').read_text()+';',encoding='utf-8')
print(json.dumps(flow_coverage,indent=2))

{
  "mapped_people": 678,
  "mapped_city_pairs": 972,
  "mapped_education_records": 1841,
  "mapped_appointment_records": 1042,
  "unmapped_education_records": 11,
  "unmapped_appointment_records": 619
}


In [10]:
flow_rows[["education_place","work_place","individuals"]].head(15)

,education_place,work_place,individuals
27,"Beijing, Beijing, China","Beijing, Hebei, China",69
8,"New York City, New York, United States","New York, New York, United States",43
64,"Beijing, Beijing, China","New York, New York, United States",33
33,"Shanghai, Shanghai, China","New York, New York, United States",29
169,"Cambridge, Massachusetts, United States","New York, New York, United States",21
40,"Beijing, Beijing, China","Kunming, Yunnan, China",19
9,"Cambridge, Massachusetts, United States","Beijing, Hebei, China",19
14,"Shanghai, Shanghai, China","Shanghai, Jiangsu, China",19
0,"Ann Arbor, Michigan, United States","Ann Arbor, Michigan, United States",17
45,"Nanjing, Jiangsu, China","Nanjing, Jiangsu, China",16


## 7. Offline HTML exports and reference cases

Rebuild the two network views, direct institution Sankey and education-to-work flow map. All views support independent university/employer countries, gender, discipline, source education Level, appointment status, sector, position level and period selection including unknown years. Networks also support centrality ranges and physics controls. `assets/engine.js` provides the exact browser data algorithms; `assets/ui_helpers.js` contains tooltip and physics controls. HTML exports embed required libraries and data for offline use.

In [11]:
def records(d):return json.loads(d.to_json(orient='records',force_ascii=False))
payload={'people':records(people),'education':records(edu),'jobs':records(jobs),'summary':summary}
(ROOT/'assets/combined_data.js').write_text('window.COMBINED='+json.dumps(payload,ensure_ascii=False,separators=(',',':'))+';',encoding='utf-8')
export(pd.DataFrame(summary.items(),columns=['metric','value']),'summary_metrics')
from render_combined import build_pages,write_methods
build_pages(ROOT);write_methods(ROOT,summary)
from validate_reference import prepare_reference
prepare_reference(ROOT,people,edu,jobs,nodes,projection_nodes,summary)
checks={'person_key_unique':people.NameID.is_unique,'education_rows_preserved':len(edu)==len(education),'appointment_rows_preserved':len(jobs)==len(career),'three_mode_nodes':G.number_of_nodes(),'three_mode_edges':G.number_of_edges(),'projection_nodes':P.number_of_nodes(),'projection_edges':P.number_of_edges(),'sankey_direct_pairs':len(direct_links),'map_people':flow_coverage['mapped_people']}
(ROOT/'validation.json').write_text(json.dumps(checks,indent=2))
print('Created networks, direct institution Sankey, education-to-work flow map and reproducible test references.')

Created networks, direct institution Sankey, education-to-work flow map and reproducible test references.


## 8. Independently verify filters, networks, Sankey, map and controls

Compare exact JavaScript calculations with pandas/NetworkX references across all filter scenarios. Checks include university/discipline/education-level row coupling, direct Sankey pair identities/counts, Other-group deduplication, city-pair identities/coordinates and missingness, endpoint person counts, antimeridian splitting and physics pause/resume/pace calls. Tooltips are checked as text elements. Node.js is needed for these optional JavaScript tests. These are code/data/control tests; local-file browser rendering was not visually verified.

In [12]:
import shutil, subprocess
node = shutil.which("node")
if node:
    checked = subprocess.run([node,str(ROOT / "validate_engine.cjs")],capture_output=True,text=True,check=True)
    print(checked.stdout)
else:
    print("Node.js is not installed. After installing it, run: node validate_engine.cjs")

{
  "filter_scenarios_passed": 39,
  "network_modes_compared_with_networkx": 4,
  "projection_shared_people_and_weights_verified": true,
  "centrality_threshold_scenarios_passed": 16,
  "direct_sankey_filter_scenarios_passed": 39,
  "grouped_sankey_deduplication_scenarios_passed": 78,
  "map_filter_and_coordinate_scenarios_passed": 39,
  "endpoint_counts_deduplicate_linked_people": true,
  "antimeridian_and_zero_vs_missing_coordinate_tests_passed": true,
  "education_level_row_coupling_fixture_passed": true,
  "network_tooltips_use_text_elements": true,
  "physics_pace_pause_and_resume_controls_verified": true,
  "browser_visual_test": "Not performed; local-file preview was blocked by application security in the prior analysis."
}



## Open the deliverables

- [2/3-mode individual–institution network](individual_institutions.html)
- [University–employer projection](university_employer.html)
- [Direct education institution → employer Sankey](education_employment_sankey.html)
- [Interactive education-to-work flow map](education_work_flows.html)
- [Methods, filters, limitations and tables](methods.html)

Each HTML is independently portable. Preserve the folder structure for navigation and notebook execution. The flow map reports coordinate coverage and does not infer missing locations or chronological journeys.